# API Reference

> Inputs, outputs, and common options for Rugprint.

Rugprint has a small object-oriented API. The usual workflow is:

1. prepare a dataframe;
2. choose projection pairs;
3. choose a sparse layout;
4. create a `Rugprint` object with `load(...)`;
5. call `.plot()` to get a matplotlib figure.

In [ ]:
from rugprint.core import (
    Rugprint,
    load,
    rugprint,
    load_penguins,
    default_penguins_layout,
    rank_pair_separation,
)

## `load(data, **kwargs) -> Rugprint`

Create a reusable `Rugprint` object. This is the recommended entry point.

In [ ]:
penguins = load_penguins()
projections, layout = default_penguins_layout()

rp = load(
    penguins,
    projections=projections,
    layout=layout,
    group="species",
    highlight=0,
)

rp

### Required Inputs

`data`
: A pandas dataframe or dataframe-like object.

`projections`
: A list of `(x_column, y_column)` pairs. Each pair becomes one scatterplot panel.

`layout`
: A dictionary mapping each projection to a `(row, column)` coordinate. Coordinates do not need to form a complete grid; empty cells stay empty.

### Optional Data Inputs

`group`
: Column name used to color observations and create a figure-level legend.

`highlight`
: Integer row position or dataframe index label. The highlighted observation is emphasized in every projection.

## `Rugprint.plot(...) -> matplotlib.figure.Figure`

Draw the projection-rug map. The returned object is a matplotlib `Figure`, so you can save or further customize it.

In [ ]:
fig = rp.plot(
    title="Palmer Penguins projection-rug map",
)

Common plot options:

`diagram_mode=True`
: Use a low-furniture diagram style. This hides axis labels and tick labels by default.

`show_axis_labels=False`
: Show or hide compact per-panel variable labels.

`show_tick_labels=False`
: Show or hide tick labels.

`rug_edges="shared"`
: Choose where rugs are drawn. Supported values are `"shared"`, `"minimal"`, `"outer"`, or a dictionary specifying exact edges per projection.

`connect_shared_rugs=True`
: If a highlighted observation is set, draw subtle connectors between adjacent panels that share a variable.

`connector_style="dotted"`
: Style for highlighted connectors. `"dotted"` and `"dashed"` are supported, or pass a matplotlib linestyle.

`panel_gap=0.04`
: Distance between panels in diagram mode.

`rug_gap=0.01`
: Distance between a panel border and its rug marks.

`point_size`, `alpha`, `rug_alpha`, `rug_length`
: Visual tuning for points and rugs.

## `.with_highlight(highlight) -> Rugprint`

Return a copy of the object with a different highlighted observation. This is useful for tracking several individual rows through the same projection map.

In [ ]:
fig = rp.with_highlight(12).plot(
    title="A different penguin tracked through the same map",
)

## `.rank_pairs(features=None, group=None) -> pandas.DataFrame`

Rank candidate projection pairs using this object's data. If `features` is omitted, Rugprint uses the variables already present in the projection map.

In [ ]:
rp.rank_pairs()

The returned dataframe has:

`x`
: First variable in the candidate pair.

`y`
: Second variable in the candidate pair.

`mean_centroid_distance`
: Mean pairwise Euclidean distance between group centroids in that two-dimensional projection.

## `rugprint(data, **kwargs) -> matplotlib.figure.Figure`

A one-shot wrapper for quick scripts. Internally, this creates a `Rugprint` object and immediately calls `.plot()`.

In [ ]:
fig = rugprint(
    penguins,
    projections=projections,
    layout=layout,
    group="species",
    highlight=0,
    title="One-shot Rugprint call",
)

## `rank_pair_separation(data, features, group) -> pandas.DataFrame`

Standalone helper for ranking all feature pairs by mean between-group centroid separation.

In [ ]:
features = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]

rank_pair_separation(penguins, features, group="species")

## Edge Specification Examples

Use `rug_edges` when you want direct control over where projection rugs appear.

In [ ]:
rug_edges = {
    ("bill_length_mm", "bill_depth_mm"): ("bottom", "left"),
    ("bill_length_mm", "flipper_length_mm"): ("top", "left"),
    ("body_mass_g", "flipper_length_mm"): ("bottom", "right"),
    ("body_mass_g", "bill_depth_mm"): ("bottom", "left"),
}

fig = rp.plot(rug_edges=rug_edges, title="Custom rug edges")

## Credit

Rugprint is inspired by Edward R. Tufte's projection/rug display idea in *The Visual Display of Quantitative Information* (Graphics Press, 1983).